# Train and Deploy a Machine Learning Model using AWS SageMaker
**Third Year B.Tech (AI & DS) Assignment**

This notebook demonstrates an end-to-end ML lifecycle on AWS SageMaker:
1. Data preparation & upload to Amazon S3 (stratified 80/20 split of Breast Cancer Wisconsin dataset).
2. Training a Random Forest classifier in **SageMaker Script Mode** with managed instances (`ml.m5.large`).
3. Capturing training metrics with CloudWatch regular expressions.
4. Deploying the model artifact to a real-time HTTPS SageMaker endpoint (`ml.t2.medium`).
5. Generating inferences and evaluating model performance.

In [ ]:
# Install SageMaker Python SDK v2 if not already installed
!pip install --quiet "sagemaker<3" pandas scikit-learn boto3

## 1. Environment & S3 Configuration

In [ ]:
import sagemaker
import boto3
import os
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

session = sagemaker.Session()
region = session.boto_region_name

# Set your S3 bucket and IAM role
ROLL_NO = "55"
bucket = f"ty-aids-sagemaker-{ROLL_NO}"
prefix = "breast-cancer"

# If running inside SageMaker Studio, you can get the role dynamically:
try:
    role = sagemaker.get_execution_role()
except ValueError:
    role = "arn:aws:iam::043262084819:role/SageMakerAssignmentRole"

print(f"AWS Region: {region}")
print(f"S3 Bucket:  {bucket}")
print(f"Role ARN:   {role}")

## 2. Prepare & Upload Data to S3

In [ ]:
# Load Breast Cancer Wisconsin dataset
data = load_breast_cancer(as_frame=True)
df = data.frame  # 30 numerical features + 1 target

print(f"Dataset shape: {df.shape}")
print("Class distribution:", df['target'].value_counts().to_dict())

# Stratified 80/20 train/test split
train, test = train_test_split(
    df,
    test_size=0.2,
    stratify=df["target"],
    random_state=42
)

# Save as CSV
os.makedirs("data", exist_ok=True)
train.to_csv("data/train.csv", index=False)
test.to_csv("data/test.csv", index=False)

# Upload to S3
train_s3 = session.upload_data("data/train.csv", bucket=bucket, key_prefix=f"{prefix}/train")
test_s3 = session.upload_data("data/test.csv", bucket=bucket, key_prefix=f"{prefix}/test")

print(f"Train uploaded to: {train_s3}")
print(f"Test uploaded to:  {test_s3}")

## 3. Create SageMaker Training Script (`train.py`)
We write the training script directly to `train.py` using Jupyter's `%%writefile` magic.

In [ ]:
%%writefile train.py
import argparse, os, joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)

def model_fn(model_dir):
    """Used by the SageMaker Scikit-learn inference container to load the model."""
    return joblib.load(os.path.join(model_dir, "model.joblib"))

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--n-estimators", type=int, default=200)
    parser.add_argument("--max-depth", type=int, default=8)
    parser.add_argument("--model-dir", default=os.environ.get("SM_MODEL_DIR", "./model"))
    parser.add_argument("--train", default=os.environ.get("SM_CHANNEL_TRAIN", "./data"))
    parser.add_argument("--test", default=os.environ.get("SM_CHANNEL_TEST", "./data"))
    args = parser.parse_args()

    tr = pd.read_csv(os.path.join(args.train, "train.csv"))
    te = pd.read_csv(os.path.join(args.test, "test.csv"))

    X_tr, y_tr = tr.drop("target", axis=1), tr["target"]
    X_te, y_te = te.drop("target", axis=1), te["target"]

    model = RandomForestClassifier(
        n_estimators=args.n_estimators,
        max_depth=args.max_depth,
        random_state=42
    )
    model.fit(X_tr, y_tr)

    pred = model.predict(X_te)
    prob = model.predict_proba(X_te)[:, 1]

    print(f"test_accuracy={accuracy_score(y_te, pred):.4f}")
    print(f"test_precision={precision_score(y_te, pred):.4f}")
    print(f"test_recall={recall_score(y_te, pred):.4f}")
    print(f"test_f1={f1_score(y_te, pred):.4f}")
    print(f"test_auc={roc_auc_score(y_te, prob):.4f}")

    os.makedirs(args.model_dir, exist_ok=True)
    joblib.dump(model, os.path.join(args.model_dir, "model.joblib"))
    print("Model saved successfully!")

## 4. Launch SageMaker Training Job

In [ ]:
from sagemaker.sklearn.estimator import SKLearn

metric_definitions = [
    {"Name": "test:accuracy", "Regex": "test_accuracy=([0-9\\.]+)"},
    {"Name": "test:precision", "Regex": "test_precision=([0-9\\.]+)"},
    {"Name": "test:recall", "Regex": "test_recall=([0-9\\.]+)"},
    {"Name": "test:f1", "Regex": "test_f1=([0-9\\.]+)"},
    {"Name": "test:auc", "Regex": "test_auc=([0-9\\.]+)"},
]

estimator = SKLearn(
    entry_point="train.py",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version="1.2-1",
    py_version="py3",
    hyperparameters={"n-estimators": 200, "max-depth": 8},
    metric_definitions=metric_definitions,
    output_path=f"s3://{bucket}/{prefix}/output"
)

print("Launching SageMaker Training Job...")
estimator.fit({"train": train_s3, "test": test_s3})

## 5. Deploy Real-Time Endpoint
> **Cost Alert**: SageMaker real-time endpoints bill per hour continuously while in service. Once you finish testing and screenshotting, execute the cleanup step at the bottom of the notebook.

In [ ]:
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.t2.medium",
    endpoint_name="breast-cancer-rf-endpoint"
)
print("Endpoint deployed and InService!")

## 6. Test Endpoint Predictions

In [ ]:
sample = test.drop("target", axis=1).iloc[:5].values
ground_truth = test["target"].iloc[:5].values

predictions = predictor.predict(sample)
print("Predictions (1=Benign, 0=Malignant):", predictions)
print("Ground Truth Labels:               ", ground_truth.tolist())

## 7. Clean Up SageMaker Endpoint
Delete the endpoint to avoid incurring hourly charges.

In [ ]:
# Execute this after testing Lambda and API Gateway!
# predictor.delete_endpoint()
# print("SageMaker endpoint deleted successfully.")